# Fig 5C - biological convergence (positive)

Blood APA hits recover the recurrent cytoskeletal/RHO-GTPase machinery of the cortical arms (Figs 1-3) plus a peripheral immune signature; many also drive the classifier. Cross-tissue _set-level_ overlap is not statistically enriched (hypergeometric n.s.; Fig S5e) - shown here as named, illustrative recurrent targets.


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment" / "downstr.yml").is_file())
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")

In [ ]:
import sys, warnings

warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt

sys.path.insert(0, f"{REPO_ROOT}/figures/style")
import paper_fig_style as S

S.apply_rcparams()
PANEL = "fig5c_convergence"
B = f"{DATA_ROOT}/3utr/Blood/2025_analyses/DTU_curated_sexadj/"
KB = f"{DATA_ROOT}/3utr/organoids/knoblich/utrcalling/"
LB = f"{DATA_ROOT}/3utr/Blood/2025_analyses/LOBOCV_genelevel_curated_sexadj_ridge/"
g = pd.read_csv(B + "dtu_unadj_gene.csv")
f = pd.read_csv(B + "dtu_unadj_feature.csv")
f["symbol"] = f.feature_id.astype(str).str.split("-").str[0].str.split("_").str[0]
g["symbol"] = g.gene_id.map(
    f.groupby("gene_id")["symbol"].agg(
        lambda s: s.mode().iat[0] if len(s.mode()) else s.iloc[0]
    )
)
dl = pd.read_csv(B + "gene_dpsi_curated.csv").set_index("gene_id")["dpsi"]
g["dpsi"] = g.gene_id.map(dl)
hit = g[g.adj_pvalue < 0.05].copy()
H = hit.set_index("symbol")
# classifier features (top discriminative APA genes)
feat = pd.read_csv(LB + "feature_stability.csv")
feat["symbol"] = feat.feature_id.astype(str).str.split("-").str[0].str.split("_").str[0]
featset = set(feat.loc[feat.prop >= 0.8, "symbol"])
# cortical hits across Figs 1-3 (recurrent targets)
fig2 = set()
for lin in ["excitatory", "inhibitory"]:
    d = pd.read_csv(KB + f"composite_ranking/composite_ranking_optimized2_{lin}.csv")
    fig2 |= set(d.loc[d.padj < 0.05, "gene_name"].dropna())
import glob

fig3 = set()
for fp in glob.glob(
    KB + "dirmult_treatment_results/*/main_only/drimseq_gene_*_treatment_main.csv"
):
    d = pd.read_csv(fp)
    pc = "adj_pvalue" if "adj_pvalue" in d.columns else "padj"
    fig3 |= set(d.loc[d[pc] < 0.05, "gene_id"].dropna())
fig1 = set(
    pd.read_csv(
        KB + "orthologs_results/orthologs_significant_both.csv"
    ).human_name.dropna()
)
cortex = fig1 | fig2 | fig3
# functional themes recurrent across the study (curated by standard gene annotation)
CYTO = [
    "CAPZB",
    "IQGAP1",
    "NEDD9",
    "SSH2",
    "WIPF1",
    "RHOA",
    "EZR",
    "MTSS1",
    "ELMO1",
    "RIPOR2",
    "TMOD3",
    "RAB8A",
]
IMM = [
    "TREM1",
    "ADAM10",
    "ICAM1",
    "IRF1",
    "IFNAR1",
    "LCP2",
    "NFKBIB",
    "CHST15",
    "CASP8",
    "TNFAIP1",
    "MPEG1",
    "PARP9",
]


def rows(lst):
    out = []
    for s in lst:
        if s in H.index:
            out.append(
                (
                    s,
                    (
                        float(H.loc[s, "dpsi"])
                        if np.ndim(H.loc[s, "dpsi"]) == 0
                        else float(np.atleast_1d(H.loc[s, "dpsi"])[0])
                    ),
                    s in featset,
                    s in cortex,
                )
            )
    return out


CY = rows(CYTO)
IM = rows(IMM)
print("cytoskeletal/RHO blood hits shown:", [r[0] for r in CY])
print("immune blood hits shown:", [r[0] for r in IM])
print(
    "of",
    len(hit),
    "hits:",
    sum(s in CYTO for s in H.index),
    "cytoskeletal,",
    sum(s in IMM for s in H.index),
    "immune (curated themes)",
)

In [ ]:
data = [
    ("Cytoskeleton / RHO-GTPase", "#2166ac", CY),
    ("Immune / inflammation", "#b2182b", IM),
]
nrow = sum(len(d[2]) for d in data) + len(data)
fig, ax = plt.subplots(figsize=(S.HALF_W_PX / 300, S.mm_to_px(4.6 * nrow + 8) / 300))
y = 0
yt = []
yl = []
for theme, col, rws in data:
    rws = sorted(rws, key=lambda r: r[1])
    for s, dlen, isfeat, iscx in rws:
        ax.plot([0, dlen], [y, y], color=col, lw=1.1, zorder=1)
        ax.scatter(dlen, y, s=26, color=col, zorder=2, edgecolors="none")
        mk = ""
        if isfeat:
            mk += "*"  # drives the classifier
        if iscx:
            mk += "+"  # also a cortical (Fig 1-3) APA hit
        lab = ("$\\it{%s}$" % s) + (("  " + mk) if mk else "")
        yt.append(y)
        yl.append(lab)
        y += 1
    # theme header gap
    ax.text(ax.get_xlim()[0] if False else 0, y + 0.1, "", fontsize=1)
    y += 1
ax.axvline(0, color="grey", lw=0.6, zorder=0)
ax.set_yticks(yt)
ax.set_yticklabels(yl, fontsize=S.FONT_TICK - 0.5)
for t, (theme, col, rws) in zip([yt[0]], [data[0]]):
    pass
ax.set_xlabel(r"blood $\Delta\psi$ (ASD-CTRL; + = lengthening)", fontsize=S.FONT_LABEL)
ax.set_ylim(-0.8, y - 0.4)
ax.tick_params(labelsize=S.FONT_TICK, length=0)
for sp in ("top", "right", "left"):
    ax.spines[sp].set_visible(False)
# theme color legend + marker key
from matplotlib.lines import Line2D

leg = [
    Line2D([0], [0], color="#2166ac", lw=3, label="cytoskeleton / RHO"),
    Line2D([0], [0], color="#b2182b", lw=3, label="immune / inflammation"),
]
ax.legend(handles=leg, fontsize=S.FONT_LEGEND, frameon=False, loc="lower right")
ax.set_title(
    "Blood APA hits recover the study's cytoskeletal/RHO axis\n+ a peripheral immune signature   (* drives classifier; + also cortical hit)",
    fontsize=S.FONT_ANNOT - 0.5,
    pad=4,
)
fig.subplots_adjust(left=0.30, right=0.97, bottom=0.12, top=0.90)
S.save_and_show(fig, PANEL, figure="Fig5")

### Variant: third-height half-width (two-column cytoskeletal | immune)


In [ ]:
# --- fig5c variant: third-height half-width, two-column (cytoskeletal | immune) ---
def _fig5c_twocol(figsize, save_name):
    fig, (axL, axR) = plt.subplots(1, 2, figsize=figsize, sharex=True)
    for ax, (theme, col, rws, rightlab) in zip(
        [axL, axR],
        [
            ("Cytoskeleton / RHO-GTPase", "#2166ac", CY, False),
            ("Immune / inflammation", "#b2182b", IM, True),
        ],
    ):
        rws = sorted(rws, key=lambda r: r[1])
        yt = []
        yl = []
        for y, (s, dlen, isfeat, iscx) in enumerate(rws):
            ax.plot([0, dlen], [y, y], color=col, lw=1.0, zorder=1)
            ax.scatter(dlen, y, s=16, color=col, zorder=2, edgecolors="none")
            mk = ("*" if isfeat else "") + ("+" if iscx else "")
            yt.append(y)
            yl.append(("$\\it{%s}$" % s) + (("  " + mk) if mk else ""))
        ax.axvline(0, color="grey", lw=0.6, zorder=0)
        ax.set_yticks(yt)
        ax.set_yticklabels(yl, fontsize=S.FONT_TICK - 0.5)
        if rightlab:
            ax.yaxis.tick_right()
        ax.set_ylim(-0.7, len(rws) - 0.3)
        ax.tick_params(labelsize=S.FONT_TICK, length=0)
        ax.set_title(theme, fontsize=S.FONT_ANNOT - 0.5, color=col, pad=3)
        for sp in ("top", "right", "left"):
            ax.spines[sp].set_visible(False)
    fig.supxlabel(
        r"blood $\Delta\psi$ (ASD-CTRL; + = lengthening)", fontsize=S.FONT_LABEL, y=0.09
    )
    fig.text(
        0.5,
        0.015,
        "* drives classifier    + also cortical (Fig 1-3) hit",
        fontsize=S.FONT_LEGEND - 0.5,
        ha="center",
        color="0.35",
    )
    fig.subplots_adjust(left=0.19, right=0.81, bottom=0.20, top=0.90, wspace=0.60)
    S.save_and_show(fig, save_name, figure="Fig5")


_fig5c_twocol((S.HALF_W_PX / 300, S.THIRD_H_PX / 300), "fig5c_convergence__third_half")